# 07 — Pooled Yoko + Yangambi: probability-aware Musanga satellite modelling

This revision removes the single-threshold bottleneck between UAV inference and satellite modelling.

**Primary question:** how well can observation-year satellite information predict UAV-derived Musanga signal when the UAV reference is represented in three different ways?

Three paired UAV targets are built directly from the native U-Net probability rasters:
1. **balanced_binary** — fractional cover after the original site-specific operational threshold (Yoko 0.008; YB_Carlier 0.001);
2. **mean_probability** — mean raw U-Net Musanga probability, with no thresholding;
3. **high_confidence** — fractional cover using p ≥ 0.50 as a deliberately conservative/high-score sensitivity target.

Important interpretation: `mean_probability` is called *mean Musanga probability/score* here. It should only be interpreted as expected physical fractional cover if probability calibration supports that interpretation.

The notebook reuses existing T0 satellite downloads and caches wherever possible. New probability-aware target tables, models and results are written to new directories so the earlier binary-only benchmark remains reproducible.


## 01 — Setup and configuration


In [ ]:
!pip -q install earthengine-api geemap rasterio scikit-learn scipy joblib

from pathlib import Path
import re, warnings, json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, balanced_accuracy_score
)
import ee, geemap

from google.colab import drive
DRIVE_ROOT=Path("/content/drive")
MYDRIVE=DRIVE_ROOT/"MyDrive"
if not MYDRIVE.exists() or not any(MYDRIVE.iterdir()):
    drive.mount(str(DRIVE_ROOT), force_remount=False)
else:
    print("Google Drive already mounted.")

ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"
UAV_DIR=ROOT/"data"/"multisite_uav"
OUT=ROOT/"work"/"07_pooled_yoko_yangambi"
OLD_OUT=ROOT/"work"/"multisite_embedding_fractional_cover"
WHOLE_INF=ROOT/"work"/"whole_orthomosaic_inference"
OUT.mkdir(parents=True,exist_ok=True)

EE_PROJECT="tropsedslu"
ACTIVE_SITES={"Yoko":2020,"YB_Carlier":2024}
SUPPORTS=[10,20,30,50]
MIN_UAV_COVERAGE=0.90
RANDOM_SEED=42
TEST_SIZE=0.20
N_TREES=400
FORCE_TABLES=False
FORCE_SATELLITE=False

# Three UAV targets.
# T1 uses the original operational threshold for each independently trained UAV model.
BALANCED_THRESHOLDS={"Yoko":0.008,"YB_Carlier":0.001}

# T3 is deliberately a high-score sensitivity target.
# For Yoko, p>=0.5 had 99.8% test precision and 68.7% recall.
# For YB_Carlier, 0.5 is treated as a high-score sensitivity threshold, NOT as
# a validated 99% precision threshold unless site-specific validation establishes that.
HIGH_CONF_THRESHOLDS={"Yoko":0.50,"YB_Carlier":0.50}

TARGET_COLS={
    "balanced_binary":"cover_balanced",
    "mean_probability":"mean_probability",
    "high_confidence":"cover_highconf",
}
COVER_THRESHOLDS=[0.01,0.05,0.20,0.50]

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Output:",OUT)


## 02 — Locate both probability rasters and operational binary masks


In [ ]:
def first_existing(candidates):
    for p in candidates:
        p=Path(p)
        if p.exists():
            return p
    return None

rows=[]
for site,year in ACTIVE_SITES.items():
    binary_hits=sorted(UAV_DIR.glob(f"{site}_musanga_binary_mask_threshold_*.tif"))
    if len(binary_hits)!=1:
        raise RuntimeError(f"{site}: expected one operational binary mask, found {len(binary_hits)}")
    binary=binary_hits[0]

    prob=first_existing([
        UAV_DIR/f"{site}_musanga_probability_uint16.tif",
        WHOLE_INF/"musanga_probability_uint16.tif" if site=="Yoko" else UAV_DIR/"__missing__",
    ])
    if prob is None:
        raise FileNotFoundError(f"{site}: probability raster not found")

    m=re.search(r"threshold_([0-9.]+)\.tif$",binary.name)
    with rasterio.open(prob) as ds:
        b=transform_bounds(ds.crs,"EPSG:4326",*ds.bounds,densify_pts=21)
        rows.append({
            "site":site,"obs_year":year,
            "probability_path":str(prob),"binary_path":str(binary),
            "operational_threshold":float(m.group(1)),
            "configured_balanced_threshold":BALANCED_THRESHOLDS[site],
            "high_conf_threshold":HIGH_CONF_THRESHOLDS[site],
            "uav_res_m":abs(ds.transform.a),
            "dtype":ds.dtypes[0],"nodata":ds.nodata,
            "width":ds.width,"height":ds.height,
            "lon_min":b[0],"lat_min":b[1],"lon_max":b[2],"lat_max":b[3]
        })

inventory=pd.DataFrame(rows)
display(inventory)
assert set(inventory.site)==set(ACTIVE_SITES)
for r in inventory.itertuples(index=False):
    assert abs(r.operational_threshold-r.configured_balanced_threshold)<1e-12, (
        f"{r.site}: filename threshold {r.operational_threshold} != configured "
        f"{r.configured_balanced_threshold}"
    )
print("Probability rasters found for both active sites. Mos remains untouched.")


## 03 — Audit/decode the uint16 probability rasters

The probability encoding is inferred and then checked against the already-saved operational binary mask. This prevents silently assuming an incorrect uint16 scale.


In [ ]:
def probability_scale(path, binary_path, threshold):
    """Infer uint16 encoding and verify it against the saved operational binary mask."""
    candidates=[65535.0,10000.0,1000.0,1.0]
    with rasterio.open(path) as ps, rasterio.open(binary_path) as bs:
        if ps.crs!=bs.crs or ps.transform!=bs.transform or ps.shape!=bs.shape:
            raise RuntimeError(f"Probability/binary grids differ: {Path(path).name}")
        h,w=ps.height,ps.width
        # Deterministic sparse windows avoid loading multi-GB rasters.
        rng=np.random.default_rng(RANDOM_SEED)
        rr=rng.integers(0,h,size=12)
        cc=rng.integers(0,w,size=12)
        raw=[]; hard=[]
        for r,c in zip(rr,cc):
            win=rasterio.windows.Window(max(0,c-128),max(0,r-128),256,256)
            a=ps.read(1,window=win)
            b=bs.read(1,window=win)
            valid=np.isfinite(a)
            if ps.nodata is not None: valid &= (a!=ps.nodata)
            valid &= np.isin(b,[0,1])
            if valid.any():
                raw.append(a[valid].astype(np.float64))
                hard.append(b[valid].astype(np.uint8))
        raw=np.concatenate(raw); hard=np.concatenate(hard)
        if np.issubdtype(raw.dtype,np.floating) and np.nanmax(raw)<=1.000001:
            candidates=[1.0]
        scores=[]
        for scale in candidates:
            p=np.clip(raw/scale,0,1)
            pred=(p>=threshold).astype(np.uint8)
            scores.append((float(np.mean(pred==hard)),scale,float(np.nanmax(p)),
                           float(np.nanmedian(p)),float(np.nanpercentile(p,99))))
        scores=sorted(scores,reverse=True)
        best=scores[0]
        if best[0]<0.98:
            raise RuntimeError(
                f"Could not verify probability encoding for {path}; best binary agreement={best[0]:.4f}"
            )
        return best[1],scores

audit_rows=[]
PROB_SCALES={}
for r in inventory.itertuples(index=False):
    scale,scores=probability_scale(r.probability_path,r.binary_path,r.operational_threshold)
    PROB_SCALES[r.site]=scale
    with rasterio.open(r.probability_path) as ds:
        audit_rows.append({
            "site":r.site,"probability_scale":scale,"dtype":ds.dtypes[0],
            "nodata":ds.nodata,"operational_threshold":r.operational_threshold,
            "binary_agreement":scores[0][0],
            "sample_p_max":scores[0][2],"sample_p_median":scores[0][3],
            "sample_p99":scores[0][4]
        })
probability_audit=pd.DataFrame(audit_rows)
display(probability_audit)
assert (probability_audit.binary_agreement>=0.98).all()
print("Decoded probability scales:",PROB_SCALES)


## 04 — Reuse/download observation-year satellite predictors

Exactly the same T0-only embedding and Sentinel-2 design as the previous Notebook 07 is retained. Existing compatible downloads are reused.


In [ ]:
EMB_BANDS=[f"A{i:02d}" for i in range(64)]
S2_BANDS=["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]
S2_FEATURES=S2_BANDS+["NDVI","NDMI","NBR","RENDVI"]

def roi_for_mask(mask_path):
    with rasterio.open(mask_path) as ds:
        b=transform_bounds(ds.crs,"EPSG:4326",*ds.bounds,densify_pts=21)
    return ee.Geometry.Rectangle(list(b),geodesic=False)

def valid_tif(path,count=None):
    try:
        with rasterio.open(path) as ds:
            return ds.width>0 and ds.height>0 and (count is None or ds.count==count)
    except Exception:
        return False

def embedding_path(site,year):
    old=OLD_OUT/"embeddings"/f"{site}_embedding_{year}_10m.tif"
    new=OUT/"embeddings"/f"{site}_embedding_{year}_10m.tif"
    if old.exists() and valid_tif(old,64): return old
    return new

def get_embedding(site,mask_path,year):
    out=embedding_path(site,year)
    out.parent.mkdir(parents=True,exist_ok=True)
    if out.exists() and valid_tif(out,64) and not FORCE_SATELLITE:
        print("Reusing",out.name)
        return out
    roi=roi_for_mask(mask_path)
    col=(ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
         .filterDate(f"{year}-01-01",f"{year+1}-01-01").filterBounds(roi))
    if col.size().getInfo()==0:
        raise RuntimeError(f"No embedding for {site} {year}")
    print("Downloading embedding",site,year)
    geemap.ee_export_image(
        col.mosaic().select(EMB_BANDS),filename=str(out),
        scale=10,region=roi,file_per_band=False
    )
    if not valid_tif(out,64):
        raise RuntimeError(f"Embedding export failed: {out}")
    return out

def mask_s2(img):
    scl=img.select("SCL")
    clear=(scl.neq(0).And(scl.neq(1)).And(scl.neq(3))
           .And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    r=img.select(S2_BANDS).multiply(0.0001)
    out=(r.addBands([
        r.normalizedDifference(["B8","B4"]).rename("NDVI"),
        r.normalizedDifference(["B8","B11"]).rename("NDMI"),
        r.normalizedDifference(["B8","B12"]).rename("NBR"),
        r.normalizedDifference(["B8A","B5"]).rename("RENDVI")
    ]).updateMask(clear))
    # Critical: preserve acquisition time so filterDate still works after map(mask_s2).
    return out.copyProperties(img, ["system:time_start", "system:index"])

def s2_path(site,year):
    return OUT/"sentinel2"/f"{site}_S2_quarterly_T0_{year}_10m.tif"

def get_s2(site,mask_path,year):
    out=s2_path(site,year)
    out.parent.mkdir(parents=True,exist_ok=True)
    if out.exists() and valid_tif(out,56) and not FORCE_SATELLITE:
        print("Reusing",out.name)
        return out
    roi=roi_for_mask(mask_path)
    # Same ordering/logic as the working Notebook 06 M05 implementation.
    # Filter the source collection first, then map the cloud-mask function.
    col=(ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi)
         .filterDate(f"{year}-01-01",f"{year+1}-01-01")
         .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE",80)).map(mask_s2))
    print(f"{site} {year}: {col.size().getInfo()} total candidate S2 scenes")
    ims=[]
    for yy in [year]:
        for q,month in enumerate([1,4,7,10],1):
            qcol=col.filterDate(ee.Date.fromYMD(yy,month,1),ee.Date.fromYMD(yy,month,1).advance(3,"month"))
            n=qcol.size().getInfo()
            print(f"{site} {yy} Q{q}: {n} S2 scenes")
            if n==0:
                raise RuntimeError(f"No S2 scenes for {site} {yy} Q{q}")
            names=[f"S2_{yy}_Q{q}_{b}" for b in S2_FEATURES]
            ims.append(qcol.median().select(S2_FEATURES).rename(names))
    stack=ee.Image.cat(ims).toFloat()
    print("Downloading S2",site,year)
    geemap.ee_export_image(stack,filename=str(out),scale=10,region=roi,file_per_band=False)
    if not valid_tif(out,56):
        raise RuntimeError(f"S2 export failed: {out}")
    return out

satellite={}
for row in inventory.itertuples(index=False):
    mask=Path(row.probability_path); y=int(row.obs_year)
    satellite[row.site]={
        "emb":get_embedding(row.site,mask,y),
        "s2":get_s2(row.site,mask,y)
    }

print("Satellite caches ready.")

## 05 — Build three UAV targets directly from native probability pixels

For every requested 10/20/30/50 m support, aggregation starts from the native UAV probability raster. No 10 m target is re-aggregated to coarser scales. Satellite predictors are aligned once at 10 m and reused in memory.


In [ ]:
EMB_COLS=[f"E_T0_A{j:02d}" for j in range(64)]
S2_COLS=[f"S2_{j:03d}" for j in range(56)]
ALL_FEATURE_COLS=EMB_COLS+S2_COLS
FEATURE_SETS={"Embeddings":EMB_COLS,"Sentinel2":S2_COLS,"Fusion":ALL_FEATURE_COLS}

def target_grid(ref_path,support):
    with rasterio.open(ref_path) as ref:
        factor=int(round(support/10))
        h=int(np.ceil(ref.height/factor)); w=int(np.ceil(ref.width/factor))
        tr=ref.transform*rasterio.Affine.scale(factor,factor)
        return (h,w),ref.crs,tr

def aggregate_probability_targets(prob_path,shape,crs,tr,scale,balanced_thr,high_thr):
    """
    Aggregate native UAV probabilities DIRECTLY to the requested support.
    Returns:
      mean_probability = mean raw U-Net probability among valid UAV pixels
      cover_balanced   = fraction p >= original operational threshold
      cover_highconf   = fraction p >= high-confidence score threshold
      uav_valid_fraction = fraction of target cell covered by valid UAV probability pixels
    """
    with rasterio.open(prob_path) as src:
        raw=src.read(1)
        valid=np.isfinite(raw)
        if src.nodata is not None:
            valid &= (raw!=src.nodata)
        p=np.zeros(raw.shape,np.float32)
        p[valid]=np.clip(raw[valid].astype(np.float32)/float(scale),0,1)

        layers={
            "mean_probability":np.where(valid,p,0).astype(np.float32),
            "cover_balanced":np.where(valid,p>=balanced_thr,0).astype(np.float32),
            "cover_highconf":np.where(valid,p>=high_thr,0).astype(np.float32),
        }
        valid_f=valid.astype(np.float32)

        den=np.zeros(shape,np.float32)
        reproject(valid_f,den,
                  src_transform=src.transform,src_crs=src.crs,src_nodata=None,
                  dst_transform=tr,dst_crs=crs,dst_nodata=0.0,
                  resampling=Resampling.average,init_dest_nodata=True)

        out={}
        for name,a in layers.items():
            num=np.zeros(shape,np.float32)
            reproject(a,num,
                      src_transform=src.transform,src_crs=src.crs,src_nodata=None,
                      dst_transform=tr,dst_crs=crs,dst_nodata=0.0,
                      resampling=Resampling.average,init_dest_nodata=True)
            z=np.full(shape,np.nan,np.float32)
            np.divide(num,den,out=z,where=den>0)
            out[name]=z
        return out,den

def align_stack_once(path,ref_path):
    with rasterio.open(ref_path) as ref:
        shape=(ref.height,ref.width); crs=ref.crs; tr=ref.transform
    with rasterio.open(path) as src:
        out=np.full((src.count,*shape),np.nan,np.float32)
        same=(src.crs==crs and src.transform.almost_equals(tr)
              and (src.height,src.width)==shape)
        if same:
            out=src.read().astype("float32")
            valid=np.isfinite(out)
            if src.nodata is not None: valid &= (out!=src.nodata)
            out[~valid]=np.nan
        else:
            for b in range(src.count):
                reproject(rasterio.band(src,b+1),out[b],
                          src_transform=src.transform,src_crs=src.crs,src_nodata=src.nodata,
                          dst_transform=tr,dst_crs=crs,dst_nodata=np.nan,
                          resampling=Resampling.bilinear,init_dest_nodata=True)
        return out

def aggregate_cube_from_10m(cube,factor,out_shape):
    if factor==1: return cube
    bands,h,w=cube.shape; oh,ow=out_shape
    ph=oh*factor; pw=ow*factor
    padded=np.full((bands,ph,pw),np.nan,np.float32)
    padded[:,:h,:w]=cube
    z=padded.reshape(bands,oh,factor,ow,factor)
    count=np.sum(np.isfinite(z),axis=(2,4))
    total=np.nansum(z,axis=(2,4))
    out=np.full((bands,oh,ow),np.nan,np.float32)
    np.divide(total,count,out=out,where=count>0)
    return out

satellite_10m={}
for r in inventory.itertuples(index=False):
    print(f"Preparing/reusing 10 m satellite cube: {r.site}")
    emb=align_stack_once(satellite[r.site]["emb"],satellite[r.site]["emb"])
    s2=align_stack_once(satellite[r.site]["s2"],satellite[r.site]["emb"])
    if emb.shape[0]!=64 or s2.shape[0]!=56:
        raise RuntimeError(f"{r.site}: expected 64 embedding + 56 S2 bands")
    satellite_10m[r.site]=np.concatenate([emb,s2],axis=0)

def build_table(r,support):
    site=r.site; y=int(r.obs_year)
    shape,crs,tr=target_grid(satellite[site]["emb"],support)
    targets,cov=aggregate_probability_targets(
        r.probability_path,shape,crs,tr,PROB_SCALES[site],
        BALANCED_THRESHOLDS[site],HIGH_CONF_THRESHOLDS[site]
    )
    factor=int(round(support/10))
    cube=aggregate_cube_from_10m(satellite_10m[site],factor,shape)

    good=(cov>=MIN_UAV_COVERAGE
          & np.all(np.isfinite(cube),axis=0)
          & np.isfinite(targets["mean_probability"])
          & np.isfinite(targets["cover_balanced"])
          & np.isfinite(targets["cover_highconf"]))
    rr,cc=np.where(good)

    # x/y are stored explicitly as lon/lat for diagnostics only.
    xs,ys=rasterio.transform.xy(tr,rr,cc,offset="center")
    lon,lat=rasterio.warp.transform(crs,"EPSG:4326",list(xs),list(ys))

    base={
        "site":np.repeat(site,len(rr)),"obs_year":np.repeat(y,len(rr)),
        "support_m":np.repeat(support,len(rr)),"row":rr,"col":cc,
        "x":np.asarray(lon),"y":np.asarray(lat),
        "uav_valid_fraction":cov[good],
        "balanced_threshold":np.repeat(BALANCED_THRESHOLDS[site],len(rr)),
        "high_conf_threshold":np.repeat(HIGH_CONF_THRESHOLDS[site],len(rr)),
        "mean_probability":targets["mean_probability"][good],
        "cover_balanced":targets["cover_balanced"][good],
        "cover_highconf":targets["cover_highconf"][good],
    }
    feats={name:cube[j][good] for j,name in enumerate(ALL_FEATURE_COLS)}
    return pd.DataFrame({**base,**feats})

tables={}
TABLE_DIR=OUT/"tables_T0_probability_targets_v2"
for support in SUPPORTS:
    parts=[]
    print(f"\n=== {support} m ===")
    for r in inventory.itertuples(index=False):
        cache=TABLE_DIR/f"{r.site}_{support}m.csv"
        cache.parent.mkdir(parents=True,exist_ok=True)
        if cache.exists() and not FORCE_TABLES:
            print("Reusing",cache.name)
            d=pd.read_csv(cache)
        else:
            print("Building probability targets:",r.site)
            d=build_table(r,support)
            d.to_csv(cache,index=False)

        req=ALL_FEATURE_COLS+list(TARGET_COLS.values())+["uav_valid_fraction"]
        valid=(np.isfinite(d[req].to_numpy(float)).all(axis=1)
               & (d["uav_valid_fraction"].to_numpy(float)>=MIN_UAV_COVERAGE))
        for c in TARGET_COLS.values():
            valid &= d[c].between(0,1).to_numpy()
        d=d.loc[valid].reset_index(drop=True)
        assert len(d)>0
        parts.append(d)

    tables[support]=pd.concat(parts,ignore_index=True)
    display(tables[support].groupby("site")[list(TARGET_COLS.values())].agg(["count","mean","median","max"]))

satellite_10m.clear()
print("\nThree-target tables ready.")


## 06 — Paired random-split benchmark

All three targets use exactly the same modelling cells and train/test indices. This isolates the effect of target definition. Embeddings, Sentinel-2 and Fusion are compared as before. RF models and held-out predictions are saved.


In [ ]:
def metrics(y,p):
    return {
        "R2":r2_score(y,p),
        "RMSE":mean_squared_error(y,p)**0.5,
        "MAE":mean_absolute_error(y,p),
        "Spearman":spearmanr(y,p).statistic,
        "bias":float(np.mean(p-y)),
        "obs_mean":float(np.mean(y)),
        "pred_mean":float(np.mean(p))
    }

continuous=[]; detection=[]
MODELS=OUT/"models_T0_three_targets"; MODELS.mkdir(parents=True,exist_ok=True)
PREDICTIONS=OUT/"predictions_T0_three_targets"; PREDICTIONS.mkdir(parents=True,exist_ok=True)

for support,d in tables.items():
    idx=np.arange(len(d))
    tr,te=train_test_split(idx,test_size=TEST_SIZE,random_state=RANDOM_SEED,
                           stratify=d.site.to_numpy())
    print(f"\n{support} m: {len(d):,} cells; SAME train/test split for all targets and predictors")

    for target_name,target_col in TARGET_COLS.items():
        ytr=d.iloc[tr][target_col].to_numpy(np.float32)
        yte=d.iloc[te][target_col].to_numpy(np.float32)

        for predictor_name,cols in FEATURE_SETS.items():
            Xtr=d.iloc[tr][cols].to_numpy(np.float32)
            Xte=d.iloc[te][cols].to_numpy(np.float32)
            assert np.isfinite(Xtr).all() and np.isfinite(Xte).all()
            model=RandomForestRegressor(
                n_estimators=N_TREES,min_samples_leaf=2,max_features="sqrt",
                n_jobs=-1,random_state=RANDOM_SEED
            )
            model.fit(Xtr,ytr)
            pred=np.clip(model.predict(Xte),0,1)

            continuous.append({
                "support_m":support,"target":target_name,"target_col":target_col,
                "predictors":predictor_name,"n_train":len(tr),"n_test":len(te),
                **metrics(yte,pred)
            })

            joblib.dump({
                "model":model,"support_m":support,"target":target_name,
                "target_col":target_col,"predictors":predictor_name,
                "feature_columns":cols,"random_seed":RANDOM_SEED,
                "test_size":TEST_SIZE,"min_uav_coverage":MIN_UAV_COVERAGE,
                "balanced_thresholds":BALANCED_THRESHOLDS,
                "high_conf_thresholds":HIGH_CONF_THRESHOLDS,
                "probability_scales":PROB_SCALES,
                "n_train":len(tr),"n_test":len(te)
            },MODELS/f"RF_{support}m_{target_name}_{predictor_name}_T0.joblib",compress=3)

            q=d.iloc[te][["site","obs_year","support_m","row","col","x","y",
                          "uav_valid_fraction",target_col]].copy()
            q["target"]=target_name; q["prediction"]=pred
            q.to_csv(PREDICTIONS/f"heldout_{support}m_{target_name}_{predictor_name}_T0.csv",index=False)

            for thr in COVER_THRESHOLDS:
                truth=(yte>thr).astype(int); hard=(pred>thr).astype(int)
                detection.append({
                    "support_m":support,"target":target_name,"predictors":predictor_name,
                    "threshold":thr,"prevalence":truth.mean(),
                    "ROC_AUC":roc_auc_score(truth,pred) if len(np.unique(truth))==2 else np.nan,
                    "PR_AUC":average_precision_score(truth,pred) if len(np.unique(truth))==2 else np.nan,
                    "F1":f1_score(truth,hard,zero_division=0),
                    "precision":precision_score(truth,hard,zero_division=0),
                    "recall":recall_score(truth,hard,zero_division=0),
                    "balanced_accuracy":balanced_accuracy_score(truth,hard) if len(np.unique(truth))==2 else np.nan
                })

continuous=pd.DataFrame(continuous)
detection=pd.DataFrame(detection)
display(continuous.sort_values(["support_m","target","predictors"]))


## 07 — Spatial CV: Fusion at 30 m and 50 m

The corrected 500 m row/column spatial blocks are used. Fold membership is generated once per support and reused for all three UAV targets, giving a paired comparison.


In [ ]:
SPATIAL_SUPPORTS=[30,50]
SPATIAL_BLOCK_M=500
SPATIAL_N_SPLITS=5

def spatial_groups(d,block_m=SPATIAL_BLOCK_M):
    support=d["support_m"].to_numpy(float)
    cells=np.maximum(1,np.round(block_m/support).astype(np.int64))
    bx=d["col"].to_numpy(np.int64)//cells
    by=d["row"].to_numpy(np.int64)//cells
    return np.array([f"{s}_{ix}_{iy}" for s,ix,iy in zip(d["site"],bx,by)],dtype=object)

spatial_rows=[]; spatial_predictions=[]
for support in SPATIAL_SUPPORTS:
    d=tables[support].reset_index(drop=True)
    groups=spatial_groups(d)
    n_groups=len(np.unique(groups))
    print(f"\n=== {support} m: {n_groups} spatial blocks ===")
    print(pd.DataFrame({"site":d.site,"group":groups}).groupby("site").group.nunique())
    gkf=GroupKFold(n_splits=SPATIAL_N_SPLITS)

    # Generate fold indices ONCE and reuse them for all three targets.
    folds=list(gkf.split(np.zeros(len(d)),groups=groups))
    X=d[FEATURE_SETS["Fusion"]].to_numpy(np.float32)

    for target_name,target_col in TARGET_COLS.items():
        y=d[target_col].to_numpy(np.float32)
        print(f"\n{support} m | {target_name}")
        for fold,(tr,te) in enumerate(folds,1):
            assert set(groups[tr]).isdisjoint(set(groups[te]))
            model=RandomForestRegressor(
                n_estimators=N_TREES,min_samples_leaf=2,max_features="sqrt",
                n_jobs=-1,random_state=RANDOM_SEED+fold
            )
            model.fit(X[tr],y[tr])
            pred=np.clip(model.predict(X[te]),0,1)
            m=metrics(y[te],pred)
            spatial_rows.append({
                "support_m":support,"target":target_name,"target_col":target_col,
                "predictors":"Fusion","fold":fold,"block_m":SPATIAL_BLOCK_M,
                "n_blocks":n_groups,"n_train":len(tr),"n_test":len(te),**m
            })
            q=d.iloc[te][["site","obs_year","support_m","row","col","x","y",
                          "uav_valid_fraction",target_col]].copy()
            q["target"]=target_name; q["fold"]=fold
            q["spatial_block"]=groups[te]; q["prediction"]=pred
            spatial_predictions.append(q)
            print(f"fold {fold}: R2={m['R2']:.3f} RMSE={m['RMSE']:.3f} "
                  f"MAE={m['MAE']:.3f} rho={m['Spearman']:.3f}")

spatial_cv=pd.DataFrame(spatial_rows)
spatial_summary=(spatial_cv.groupby(["support_m","target","predictors","block_m"])
                 .agg(R2_mean=("R2","mean"),R2_sd=("R2","std"),
                      RMSE_mean=("RMSE","mean"),RMSE_sd=("RMSE","std"),
                      MAE_mean=("MAE","mean"),MAE_sd=("MAE","std"),
                      Spearman_mean=("Spearman","mean"),Spearman_sd=("Spearman","std"),
                      bias_mean=("bias","mean"))
                 .reset_index())
display(spatial_summary)


## 08 — Save and compare


In [ ]:
RESULTS=OUT/"results_three_targets"; RESULTS.mkdir(parents=True,exist_ok=True)
continuous.to_csv(RESULTS/"random_split_continuous_metrics.csv",index=False)
detection.to_csv(RESULTS/"random_split_detection_metrics.csv",index=False)
spatial_cv.to_csv(RESULTS/"spatial_cv_fusion_30m_50m_folds.csv",index=False)
spatial_summary.to_csv(RESULTS/"spatial_cv_fusion_30m_50m_summary.csv",index=False)
pd.concat(spatial_predictions,ignore_index=True).to_csv(
    RESULTS/"spatial_cv_fusion_30m_50m_predictions.csv",index=False
)
probability_audit.to_csv(RESULTS/"probability_raster_audit.csv",index=False)

# Compact paired comparison: Fusion only, random vs spatial.
random_fusion=continuous[(continuous.predictors=="Fusion") & continuous.support_m.isin(SPATIAL_SUPPORTS)]
comparison=random_fusion.merge(
    spatial_summary,on=["support_m","target","predictors"],how="left",
    suffixes=("_random","_spatial")
)
comparison.to_csv(RESULTS/"three_target_random_vs_spatial.csv",index=False)
print("\nTHREE-TARGET RANDOM vs SPATIAL")
display(comparison[[
    "support_m","target","R2","R2_mean","R2_sd",
    "RMSE","RMSE_mean","MAE","MAE_mean","Spearman","Spearman_mean","Spearman_sd"
]].sort_values(["support_m","target"]))

provenance={
    "active_sites":ACTIVE_SITES,
    "supports_m":SUPPORTS,
    "balanced_thresholds":BALANCED_THRESHOLDS,
    "high_conf_thresholds":HIGH_CONF_THRESHOLDS,
    "probability_scales":PROB_SCALES,
    "min_uav_coverage":MIN_UAV_COVERAGE,
    "random_seed":RANDOM_SEED,
    "test_size":TEST_SIZE,
    "n_trees":N_TREES,
    "target_definitions":{
        "balanced_binary":"mean(1[p >= site operational threshold])",
        "mean_probability":"mean(raw decoded U-Net probability); call probabilistic cover only if calibration supports it",
        "high_confidence":"mean(1[p >= 0.5]); high-score sensitivity target"
    },
    "note":"Satellite caches are reused. New target tables are isolated from the old binary-only tables."
}
with open(RESULTS/"provenance.json","w") as fp:
    json.dump(provenance,fp,indent=2)

print("\nSaved all revised Notebook 07 outputs to:",RESULTS)


## 09 — Interpretation rules

- **balanced_binary** reproduces the earlier operational-threshold concept and is the bridge to the previous Notebook 07.
- **mean_probability** preserves UAV model confidence and avoids a hard threshold. Treat it as a probability/score target unless calibration has been demonstrated.
- **high_confidence** deliberately sacrifices recall to test whether removing weak UAV detections improves satellite transfer. The p ≥ 0.50 threshold is independently known to be extremely precise for Yoko; for YB_Carlier it is a high-score sensitivity threshold unless separate validation establishes its precision.
- Random-split results are development metrics. The 500 m spatial CV is the primary robustness check.
- No-data and <90% UAV-coverage cells never enter model fitting.
